In [ ]:
from __future__ import annotations

import traceback
from dataclasses import dataclass, field
from pathlib import Path
from typing import Dict, Optional, Tuple, List

import numpy as np
import pandas as pd
import tifffile

from cellpose import models
from scipy.ndimage import gaussian_laplace
from scipy import ndimage as ndi
from scipy.stats import norm

from skimage import filters, morphology
from skimage.filters import threshold_otsu
from skimage.measure import regionprops_table
from skimage.segmentation import find_boundaries
from skimage.draw import rectangle_perimeter

import imageio.v2 as imageio


# ----------------------------
# Helpers: channels + paths
# ----------------------------
def get_channel_indices(row: pd.Series, n_channels: int = 4) -> Dict[str, int]:
    """Map channel name -> channel index based on columns ch0..ch{n-1}."""
    channel_map: Dict[str, int] = {}
    for i in range(n_channels):
        name = row.get(f"ch{i}")
        if isinstance(name, str) and name.strip():
            channel_map[name.strip()] = i
    return channel_map


def build_img_lookup(img_dir: Path, pattern: str = "*.tif") -> Dict[str, Path]:
    imgs = sorted(img_dir.glob(pattern))
    return {p.name: p for p in imgs}


def add_image_paths(samplesheet: pd.DataFrame, img_lookup: Dict[str, Path]) -> pd.DataFrame:
    out = samplesheet.copy()
    out["image_path"] = out["filename"].map(lambda fn: img_lookup.get(fn))
    return out


def to_uint8(img: np.ndarray, p_lo: float = 1.0, p_hi: float = 99.5) -> np.ndarray:
    """Percentile-stretch to uint8 for visualization."""
    x = img.astype(np.float32)
    lo, hi = np.percentile(x, [p_lo, p_hi])
    if hi <= lo:
        return np.zeros_like(x, dtype=np.uint8)
    x = np.clip((x - lo) / (hi - lo), 0, 1)
    return (255 * x).astype(np.uint8)


# ----------------------------
# PNG output helpers
# ----------------------------
def square_crop_coords(ymin, xmin, ymax, xmax, H, W, pad: int = 8):
    ymin, xmin, ymax, xmax = int(ymin), int(xmin), int(ymax), int(xmax)
    ymin = max(0, ymin - pad)
    xmin = max(0, xmin - pad)
    ymax = min(H, ymax + pad)
    xmax = min(W, xmax + pad)

    h = ymax - ymin
    w = xmax - xmin
    side = max(h, w)

    cy = (ymin + ymax) // 2
    cx = (xmin + xmax) // 2

    y0 = max(0, cy - side // 2)
    x0 = max(0, cx - side // 2)
    y1 = min(H, y0 + side)
    x1 = min(W, x0 + side)

    # if we hit boundary, shift back so size stays square
    y0 = max(0, y1 - side)
    x0 = max(0, x1 - side)

    return y0, x0, y1, x1
 

def make_merged_rgb(dapi_u8, gfap_u8, lamp1_u8, prab_u8):
    """
    Pseudocolor mapping:
      DAPI -> blue
      GFAP -> green
      LAMP1 -> red
      pRAB10 -> magenta (red + blue)
    """
    H, W = dapi_u8.shape
    rgb = np.zeros((H, W, 3), dtype=np.uint8)

    # red
    rgb[..., 0] = np.maximum(rgb[..., 0], lamp1_u8)
    rgb[..., 0] = np.maximum(rgb[..., 0], prab_u8)

    # green
    rgb[..., 1] = np.maximum(rgb[..., 1], gfap_u8)

    # blue
    rgb[..., 2] = np.maximum(rgb[..., 2], dapi_u8)
    rgb[..., 2] = np.maximum(rgb[..., 2], prab_u8)

    return rgb


def cellpose_outline_from_labels(
    cell_masks: np.ndarray,
    cid: int,
    thickness: int = 2,
    mode: str = "outer",
) -> np.ndarray:
    """
    Outline mask from an existing label image (NO rerun of Cellpose).
    """
    cell_bin = (cell_masks == cid)
    outline = find_boundaries(cell_bin, mode=mode)  # bool
    if thickness and thickness > 1:
        outline = morphology.dilation(outline, footprint=morphology.disk(int(thickness)))
    return outline


def overlay_outline_yellow(gray_u8: np.ndarray, outline: np.ndarray) -> np.ndarray:
    """
    Convert grayscale uint8 to RGB and paint outline in yellow.
    """
    rgb = np.stack([gray_u8, gray_u8, gray_u8], axis=-1).copy()
    rgb[outline] = np.array([255, 255, 0], dtype=np.uint8)
    return rgb


def save_cell_pngs_for_image(
    img: np.ndarray,                 # (C,H,W)
    cell_masks: np.ndarray,          # (H,W) labels (needed for outlines)
    cell_geom_df: pd.DataFrame,      # from compute_cell_geometry
    ch_index: dict,                  # from get_channel_indices
    out_base: Path,                  # e.g., out_dir / "cell_pngs" / filename_stem
    filename_stem: str,
    pad: int = 8,
    min_cell_area: int = 0,
    bbox_on_crop_edge: bool = True,  # True => draw square crop border; False => draw exact cell bbox within crop
    write_outlines: bool = True,
    outline_thickness: int = 2,
):
    out_base.mkdir(parents=True, exist_ok=True)

    # pull channels once
    dapi = img[ch_index["DAPI"]]
    gfap = img[ch_index["GFAP"]]
    lamp1 = img[ch_index["LAMP1"]]
    prab = img[ch_index["pRAB10"]]

    H, W = dapi.shape

    # normalize whole-image to keep relative scaling consistent across crops
    dapi_u8 = to_uint8(dapi)
    gfap_u8 = to_uint8(gfap)
    lamp1_u8 = to_uint8(lamp1)
    prab_u8 = to_uint8(prab)

    for _, r in cell_geom_df.iterrows():
        cid = int(r["cell_id"])
        area = int(r["cell_area_px_geom"])
        if area < min_cell_area:
            continue

        y0, x0, y1, x1 = square_crop_coords(
            r["bbox_ymin"], r["bbox_xmin"], r["bbox_ymax"], r["bbox_xmax"],
            H=H, W=W, pad=pad
        )

        cell_dir = out_base / f"{filename_stem}_cell{cid:04d}"
        cell_dir.mkdir(parents=True, exist_ok=True)

        # grayscale crops
        dapi_crop = dapi_u8[y0:y1, x0:x1]
        gfap_crop = gfap_u8[y0:y1, x0:x1]
        lamp1_crop = lamp1_u8[y0:y1, x0:x1]
        prab_crop = prab_u8[y0:y1, x0:x1]

        imageio.imwrite(cell_dir / "DAPI_gray.png",  dapi_crop)
        imageio.imwrite(cell_dir / "GFAP_gray.png",  gfap_crop)
        imageio.imwrite(cell_dir / "LAMP1_gray.png", lamp1_crop)
        imageio.imwrite(cell_dir / "pRAB10_gray.png", prab_crop)

        # merged color crop
        merged = make_merged_rgb(dapi_crop, gfap_crop, lamp1_crop, prab_crop)
        imageio.imwrite(cell_dir / "merged_color.png", merged)

        # outlines (from existing cellpose labels)
        if write_outlines:
            outline_crop = cellpose_outline_from_labels(
                cell_masks=cell_masks[y0:y1, x0:x1],
                cid=cid,
                thickness=outline_thickness,
                mode="outer",
            )

            imageio.imwrite(cell_dir / "DAPI_outline_yellow.png",  overlay_outline_yellow(dapi_crop, outline_crop))
            imageio.imwrite(cell_dir / "GFAP_outline_yellow.png",  overlay_outline_yellow(gfap_crop, outline_crop))
            imageio.imwrite(cell_dir / "LAMP1_outline_yellow.png", overlay_outline_yellow(lamp1_crop, outline_crop))
            imageio.imwrite(cell_dir / "pRAB10_outline_yellow.png", overlay_outline_yellow(prab_crop, outline_crop))

            merged_outline = merged.copy()
            merged_outline[outline_crop] = np.array([255, 255, 0], dtype=np.uint8)
            imageio.imwrite(cell_dir / "merged_color_outline_yellow.png", merged_outline)

        # merged + bbox overlay
        overlay = merged.copy()

        if bbox_on_crop_edge:
            # draw border around crop
            rr, cc = rectangle_perimeter(
                start=(0, 0),
                end=(overlay.shape[0] - 1, overlay.shape[1] - 1),
                shape=overlay.shape[:2],
            )
        else:
            # draw exact bbox (relative to crop coords)
            by0 = int(r["bbox_ymin"]) - y0
            bx0 = int(r["bbox_xmin"]) - x0
            by1 = int(r["bbox_ymax"]) - y0 - 1
            bx1 = int(r["bbox_xmax"]) - x0 - 1
            by0 = max(0, min(by0, overlay.shape[0] - 1))
            bx0 = max(0, min(bx0, overlay.shape[1] - 1))
            by1 = max(0, min(by1, overlay.shape[0] - 1))
            bx1 = max(0, min(bx1, overlay.shape[1] - 1))

            rr, cc = rectangle_perimeter(
                start=(by0, bx0),
                end=(by1, bx1),
                shape=overlay.shape[:2],
            )

        overlay[rr, cc] = 255  # white border
        imageio.imwrite(cell_dir / "merged_color_bbox.png", overlay)


def save_cell_mask_pngs_for_image(
    cell_masks: np.ndarray,          # (H,W) labels
    cell_geom_df: pd.DataFrame,      # from compute_cell_geometry
    lys_mask: np.ndarray,            # (H,W) bool
    prab_mask: np.ndarray,           # (H,W) bool
    out_base: Path,                  # same base as per-cell outputs
    filename_stem: str,
    pad: int = 8,
    min_cell_area: int = 0,
    restrict_to_cell: bool = True,
):
    """
    Saves per-cell crop mask PNGs for lysosome + pRAB10 only.
    Writes:
      masks/lys_mask.png
      masks/prab_mask.png
    """
    out_base.mkdir(parents=True, exist_ok=True)
    H, W = cell_masks.shape

    for _, r in cell_geom_df.iterrows():
        cid = int(r["cell_id"])
        area = int(r["cell_area_px_geom"])
        if area < min_cell_area:
            continue

        y0, x0, y1, x1 = square_crop_coords(
            r["bbox_ymin"], r["bbox_xmin"], r["bbox_ymax"], r["bbox_xmax"],
            H=H, W=W, pad=pad
        )

        cell_dir = out_base / f"{filename_stem}_cell{cid:04d}"
        masks_dir = cell_dir / "masks"
        masks_dir.mkdir(parents=True, exist_ok=True)

        lys_crop = lys_mask[y0:y1, x0:x1]
        prab_crop = prab_mask[y0:y1, x0:x1]

        if restrict_to_cell:
            cell_crop = (cell_masks[y0:y1, x0:x1] == cid)
            lys_crop = lys_crop & cell_crop
            prab_crop = prab_crop & cell_crop

        imageio.imwrite(masks_dir / "lys_mask.png",  (lys_crop.astype(np.uint8) * 255))
        imageio.imwrite(masks_dir / "prab_mask.png", (prab_crop.astype(np.uint8) * 255))


# ----------------------------
# Structure segmentation
# ----------------------------
@dataclass
class StructureSegParams:
    intensity_scaling_param: Tuple[float, float]
    min_area: int
    blur_sigma: float
    log_sigma_1: float
    log_cutoff_1: float
    log_sigma_2: float
    log_cutoff_2: float
    log_sigma_3: float
    log_cutoff_3: float
    vesselness_sigma: Tuple[float, ...]
    vesselness_cutoff: float


def segment_structures_acis_style(channel_img: np.ndarray, params: StructureSegParams) -> np.ndarray:
    """
    ACIS-like segmentation:
    - normal-fit stretch + normalize
    - gaussian blur
    - multi-scale LoG blob detection + frangi vesselness
    - fill holes + remove small objects
    Returns boolean mask (H, W).
    """
    ch = channel_img.astype(float)

    m, s = norm.fit(ch.flatten())
    stretch_min = max(m - params.intensity_scaling_param[0] * s, float(ch.min()))
    stretch_max = min(m + params.intensity_scaling_param[1] * s, float(ch.max()))
    if stretch_max <= stretch_min:
        return np.zeros_like(ch, dtype=bool)

    ch_n = np.clip(ch, stretch_min, stretch_max)
    image_norm = (ch_n - stretch_min) / (stretch_max - stretch_min)

    blurred = filters.gaussian(image_norm, sigma=params.blur_sigma)

    log_1 = -1.0 * (params.log_sigma_1**2) * gaussian_laplace(blurred, sigma=params.log_sigma_1)
    log_2 = -1.0 * (params.log_sigma_2**2) * gaussian_laplace(blurred, sigma=params.log_sigma_2)
    log_3 = -1.0 * (params.log_sigma_3**2) * gaussian_laplace(blurred, sigma=params.log_sigma_3)

    log_mask = (log_1 > params.log_cutoff_1) | (log_2 > params.log_cutoff_2) | (log_3 > params.log_cutoff_3)

    vesselness = filters.frangi(
        blurred, sigmas=list(params.vesselness_sigma), black_ridges=False
    ) > params.vesselness_cutoff

    combined = log_mask | vesselness
    filled = ndi.binary_fill_holes(combined)
    cleaned = morphology.remove_small_objects(filled, min_size=params.min_area)

    return cleaned.astype(bool)


def segment_lysosomes(channel_img: np.ndarray, params: StructureSegParams) -> np.ndarray:
    """
    Updated ACIS-style lysosome segmentation.
    Returns boolean mask (H, W).
    Uses: intensity_scaling_param, blur_sigma, log_sigma_1 (as log sigma), min_area
    """
    lys_ch = channel_img.astype(float)

    m, s = norm.fit(lys_ch.flatten())
    stretch_min = max(m - params.intensity_scaling_param[0] * s, float(lys_ch.min()))
    stretch_max = min(m + params.intensity_scaling_param[1] * s, float(lys_ch.max()))
    if stretch_max <= stretch_min:
        return np.zeros_like(lys_ch, dtype=bool)

    lys_ch_n = np.clip(lys_ch, stretch_min, stretch_max)
    image_norm = (lys_ch_n - stretch_min) / (stretch_max - stretch_min)

    blurred = filters.gaussian(image_norm, sigma=params.blur_sigma)

    triangle_cutoff = filters.threshold_triangle(blurred)
    global_median_cutoff = np.percentile(blurred, 50)
    th_low_cutoff = (triangle_cutoff + global_median_cutoff) / 2.0
    img_low_level = blurred > th_low_cutoff

    img_low_level_small = morphology.remove_small_objects(img_low_level, min_size=int(params.min_area), connectivity=1)
    img_low_level_small_grow = morphology.dilation(img_low_level_small, footprint=morphology.disk(2))

    otsu_cutoff = 0.333 * filters.threshold_otsu(blurred)
    img_high_level = np.zeros_like(img_low_level_small_grow, dtype=bool)

    lab_low, num_obj = morphology.label(img_low_level_small_grow, return_num=True, connectivity=1)
    for idx in range(num_obj):
        single_obj = lab_low == (idx + 1)
        if np.count_nonzero(single_obj) == 0:
            continue
        try:
            local_otsu = filters.threshold_otsu(blurred[single_obj])
        except Exception:
            local_otsu = 0.0
        if local_otsu > otsu_cutoff:
            mask_condition = np.logical_and(blurred > 0.98 * local_otsu, single_obj)
            img_high_level[mask_condition] = True

    log_sigma = params.log_sigma_1
    log_response = -1.0 * (log_sigma**2) * gaussian_laplace(blurred, sigma=log_sigma)
    bw_extra = log_response > 0.09
    bw_extra[~img_low_level_small_grow] = False

    bw_final = np.logical_or(bw_extra, img_high_level)

    filled = ndi.binary_fill_holes(bw_final)
    labeled_filled = morphology.label(filled, connectivity=1)
    lysosome_mask = morphology.remove_small_objects(labeled_filled, min_size=int(params.min_area)) > 0

    return lysosome_mask.astype(bool)


# ----------------------------
# Object assignment per cell
# ----------------------------
def label_objects_within_cells(
    cell_masks: np.ndarray,
    obj_mask: np.ndarray,
) -> Tuple[np.ndarray, Dict[int, int]]:
    """
    For each cell ID in cell_masks, label connected components of obj_mask restricted
    to that cell. Write them into one global label image, and return mapping:
      global_obj_label -> parent_cell_id
    """
    cell_ids = np.unique(cell_masks)
    cell_ids = cell_ids[cell_ids != 0]

    labels_global = np.zeros_like(obj_mask, dtype=np.int32)
    parent_cell: Dict[int, int] = {}
    current_label = 1

    for cid in cell_ids:
        single_cell_mask = (cell_masks == cid)
        obj_in_cell = obj_mask & single_cell_mask
        labeled_in_cell, n = ndi.label(obj_in_cell)

        if n == 0:
            continue

        for ll in np.unique(labeled_in_cell)[1:]:
            labels_global[labeled_in_cell == ll] = current_label
            parent_cell[current_label] = int(cid)
            current_label += 1

    return labels_global, parent_cell


# ----------------------------
# Colocalization per cell
# ----------------------------
def compute_cell_coloc(
    cell_masks: np.ndarray,
    lys_int: np.ndarray,
    prab_int: np.ndarray,
    min_pixels: int = 50,
) -> pd.DataFrame:
    cell_ids = np.unique(cell_masks)
    cell_ids = cell_ids[cell_ids != 0]

    rows: List[dict] = []

    for cid in cell_ids:
        m = (cell_masks == cid)
        npx = int(m.sum())
        if npx < min_pixels:
            continue

        a = lys_int[m].astype(np.float64)
        b = prab_int[m].astype(np.float64)

        if a.std() == 0 or b.std() == 0:
            pearson_r = np.nan
        else:
            pearson_r = float(np.corrcoef(a, b)[0, 1])

        try:
            tA = float(threshold_otsu(a)) if np.unique(a).size > 1 else 0.0
        except Exception:
            tA = 0.0
        try:
            tB = float(threshold_otsu(b)) if np.unique(b).size > 1 else 0.0
        except Exception:
            tB = 0.0

        a_pos = a > tA
        b_pos = b > tB

        denomA = float(a[a_pos].sum())
        denomB = float(b[b_pos].sum())

        M1 = float(a[a_pos & b_pos].sum() / denomA) if denomA > 0 else np.nan
        M2 = float(b[b_pos & a_pos].sum() / denomB) if denomB > 0 else np.nan

        rows.append({
            "cell_id": int(cid),
            "cell_pixels": npx,
            "pearson_r": pearson_r,
            "manders_M1_lys_in_prab": M1,
            "manders_M2_prab_in_lys": M2,
            "threshold_cutoff_lys": tA,
            "threshold_cutoff_prab": tB,
        })

    return pd.DataFrame(rows)


def compute_cell_geometry(cell_masks: np.ndarray) -> pd.DataFrame:
    """
    Per-cell centroid + bbox (for rerun matching).
    NOTE: we name area as cell_area_px_geom to avoid collision with per_cell_area metrics.
    """
    props = regionprops_table(
        cell_masks,
        properties=("label", "area", "centroid", "bbox")
    )
    df = pd.DataFrame(props).rename(columns={
        "label": "cell_id",
        "area": "cell_area_px_geom",
        "centroid-0": "cell_centroid_y",
        "centroid-1": "cell_centroid_x",
        "bbox-0": "bbox_ymin",
        "bbox-1": "bbox_xmin",
        "bbox-2": "bbox_ymax",
        "bbox-3": "bbox_xmax",
    })
    df["cell_id"] = df["cell_id"].astype(int)
    return df


def compute_per_cell_area_metrics(
    cell_masks: np.ndarray,
    nuc_masks: np.ndarray,
    lys_mask: np.ndarray,
    prab_mask: np.ndarray,
) -> pd.DataFrame:
    """
    Per-cell segmented areas (pixels). This keeps cell_area_px as your primary area metric.
    """
    cell_ids = np.unique(cell_masks)
    cell_ids = cell_ids[cell_ids != 0]

    rows = []
    for cid in cell_ids:
        cell_m = (cell_masks == cid)

        rows.append({
            "cell_id": int(cid),
            "cell_area_px": int(cell_m.sum()),
            "nuc_area_px_in_cell": int(((nuc_masks > 0) & cell_m).sum()),
            "lys_segmented_area_px_in_cell": int((lys_mask & cell_m).sum()),
            "prab_segmented_area_px_in_cell": int((prab_mask & cell_m).sum()),
        })

    return pd.DataFrame(rows)


# ----------------------------
# Cellpose wrappers
# ----------------------------
@dataclass
class CellposeParams:
    diameter: float = 120
    batch_size: int = 32
    flow_threshold: float = 0.4
    cellprob_threshold: float = 0.0
    tile_norm_blocksize: int = 0  # 0 disables tile norm


def run_cellpose_cells(
    model: models.CellposeModel,
    dapi: np.ndarray,
    gfap: np.ndarray,
    params: CellposeParams,
) -> np.ndarray:
    stack = np.stack([dapi, gfap], axis=0)  # (C,H,W)
    masks, flows, styles = model.eval(
        stack,
        batch_size=params.batch_size,
        diameter=params.diameter,
        flow_threshold=params.flow_threshold,
        cellprob_threshold=params.cellprob_threshold,
        normalize={"tile_norm_blocksize": params.tile_norm_blocksize},
    )
    return masks


def run_cellpose_nuclei(
    model: models.CellposeModel,
    dapi: np.ndarray,
    params: CellposeParams,
) -> np.ndarray:
    masks, flows, styles = model.eval(
        dapi,
        batch_size=params.batch_size,
        diameter=params.diameter,
        flow_threshold=params.flow_threshold,
        cellprob_threshold=params.cellprob_threshold,
        normalize={"tile_norm_blocksize": params.tile_norm_blocksize},
    )
    return masks


# ----------------------------
# Per-image pipeline config
# ----------------------------
@dataclass
class PipelineConfig:
    n_channels: int = 4
    channel_names: Tuple[str, str, str, str] = ("DAPI", "GFAP", "LAMP1", "pRAB10")

    cellpose: CellposeParams = field(default_factory=CellposeParams)

    lys_params: StructureSegParams = field(default_factory=lambda: StructureSegParams(
        intensity_scaling_param=(3, 19),
        min_area=5,
        blur_sigma=1,
        log_sigma_1=3, log_cutoff_1=0.13,
        log_sigma_2=2, log_cutoff_2=0.08,
        log_sigma_3=1, log_cutoff_3=0.06,
        vesselness_sigma=(1,),
        vesselness_cutoff=0.3,
    ))

    prab_params: StructureSegParams = field(default_factory=lambda: StructureSegParams(
        intensity_scaling_param=(4, 9),
        min_area=5,
        blur_sigma=3,
        log_sigma_1=3, log_cutoff_1=0.13,
        log_sigma_2=2, log_cutoff_2=0.11,
        log_sigma_3=1, log_cutoff_3=0.09,
        vesselness_sigma=(1,),
        vesselness_cutoff=0.5,
    ))

    coloc_min_pixels: int = 50

    # PNG output controls
    write_cell_pngs: bool = True
    cell_png_pad: int = 8
    cell_png_min_area: int = 0
    bbox_on_crop_edge: bool = True

    write_outlines: bool = True
    outline_thickness: int = 2

    write_mask_pngs: bool = True
    restrict_mask_to_cell: bool = True


# ----------------------------
# Per-cell spatial metrics
# ----------------------------
def compute_per_cell_signal_spatial_metrics(
    cell_masks: np.ndarray,
    nuc_masks: np.ndarray,
    signal_mask: np.ndarray,
    signal_int: np.ndarray,
    prefix: str,
    perinuclear_radius_px: float = 10.0,
) -> pd.DataFrame:
    cell_ids = np.unique(cell_masks)
    cell_ids = cell_ids[cell_ids != 0]

    rows = []
    r = float(perinuclear_radius_px)

    for cid in cell_ids:
        cell_m = (cell_masks == cid)

        nuc_m = (nuc_masks > 0) & cell_m
        if nuc_m.sum() == 0:
            rows.append({
                "cell_id": int(cid),
                "nuc_centroid_y": np.nan,
                "nuc_centroid_x": np.nan,
                f"{prefix}_centroid_y": np.nan,
                f"{prefix}_centroid_x": np.nan,
                f"dist_nuc_to_{prefix}_centroid_px": np.nan,
                f"{prefix}_dist_mean_px": np.nan,
                f"{prefix}_dist_median_px": np.nan,
                f"{prefix}_dist_p90_px": np.nan,
                f"{prefix}_perinuclear_frac_r{int(r)}px": np.nan,
                f"{prefix}_pixels_in_cell": int((signal_mask & cell_m).sum()),
            })
            continue

        nuc_yx = np.argwhere(nuc_m)
        nuc_cy, nuc_cx = nuc_yx.mean(axis=0)

        sig_m = signal_mask & cell_m
        n_sig_px = int(sig_m.sum())
        if n_sig_px == 0:
            rows.append({
                "cell_id": int(cid),
                "nuc_centroid_y": float(nuc_cy),
                "nuc_centroid_x": float(nuc_cx),
                f"{prefix}_centroid_y": np.nan,
                f"{prefix}_centroid_x": np.nan,
                f"dist_nuc_to_{prefix}_centroid_px": np.nan,
                f"{prefix}_dist_mean_px": np.nan,
                f"{prefix}_dist_median_px": np.nan,
                f"{prefix}_dist_p90_px": np.nan,
                f"{prefix}_perinuclear_frac_r{int(r)}px": 0.0,
                f"{prefix}_pixels_in_cell": 0,
            })
            continue

        sig_coords = np.argwhere(sig_m)
        weights = signal_int[sig_m].astype(np.float64)
        wsum = weights.sum()

        if wsum > 0:
            sig_cy = float((sig_coords[:, 0] * weights).sum() / wsum)
            sig_cx = float((sig_coords[:, 1] * weights).sum() / wsum)
        else:
            sig_cy, sig_cx = sig_coords.mean(axis=0).astype(float)

        d_centroid = float(np.hypot(sig_cy - nuc_cy, sig_cx - nuc_cx))

        dy = sig_coords[:, 0].astype(np.float64) - nuc_cy
        dx = sig_coords[:, 1].astype(np.float64) - nuc_cx
        dists = np.hypot(dy, dx)

        perinu_frac = float((dists <= r).mean()) if dists.size else np.nan

        rows.append({
            "cell_id": int(cid),
            "nuc_centroid_y": float(nuc_cy),
            "nuc_centroid_x": float(nuc_cx),
            f"{prefix}_centroid_y": float(sig_cy),
            f"{prefix}_centroid_x": float(sig_cx),
            f"dist_nuc_to_{prefix}_centroid_px": d_centroid,
            f"{prefix}_dist_mean_px": float(dists.mean()),
            f"{prefix}_dist_median_px": float(np.median(dists)),
            f"{prefix}_dist_p90_px": float(np.percentile(dists, 90)),
            f"{prefix}_perinuclear_frac_r{int(r)}px": perinu_frac,
            f"{prefix}_pixels_in_cell": n_sig_px,
        })

    return pd.DataFrame(rows)


# ----------------------------
# Per-image processing
# ----------------------------
def process_one_image(
    row: pd.Series,
    cfg: PipelineConfig,
    cellpose_model: models.CellposeModel,
    png_out_dir: Optional[Path] = None,
) -> Dict[str, pd.DataFrame]:
    img_path = row.get("image_path", None)
    if img_path is None or str(img_path) == "nan":
        raise FileNotFoundError(f"No image_path for filename={row.get('filename')}")

    img_path = Path(img_path)
    img = tifffile.imread(img_path)  # expected (C,H,W)

    ch_index = get_channel_indices(row, n_channels=cfg.n_channels)
    for chname in cfg.channel_names:
        if chname not in ch_index:
            raise KeyError(f"Missing channel '{chname}' for image {img_path.name}. Got: {list(ch_index.keys())}")

    dapi = img[ch_index["DAPI"]]
    gfap = img[ch_index["GFAP"]]
    lys  = img[ch_index["LAMP1"]]
    prab = img[ch_index["pRAB10"]]

    # Cellpose
    cell_masks = run_cellpose_cells(cellpose_model, dapi, gfap, cfg.cellpose)
    nuc_masks  = run_cellpose_nuclei(cellpose_model, dapi, cfg.cellpose)

    # Geometry for rerun matching (+ bbox for PNG crops)
    cell_geom_df = compute_cell_geometry(cell_masks)
    cell_geom_df.insert(0, "filename", img_path.name)

    filename_stem = img_path.stem

    # Write per-cell PNGs (channels + merged + bbox + yellow-outline variants)
    if cfg.write_cell_pngs and png_out_dir is not None:
        save_cell_pngs_for_image(
            img=img,
            cell_masks=cell_masks,
            cell_geom_df=cell_geom_df,
            ch_index=ch_index,
            out_base=png_out_dir / filename_stem,
            filename_stem=filename_stem,
            pad=cfg.cell_png_pad,
            min_cell_area=cfg.cell_png_min_area,
            bbox_on_crop_edge=cfg.bbox_on_crop_edge,
            write_outlines=cfg.write_outlines,
            outline_thickness=cfg.outline_thickness,
        )

    # Structure masks (whole-image boolean masks)
    lys_mask  = segment_lysosomes(lys, cfg.lys_params)
    prab_mask = segment_structures_acis_style(prab, cfg.prab_params)

    # Write per-cell mask PNGs (lys/prab only)
    if cfg.write_mask_pngs and png_out_dir is not None:
        save_cell_mask_pngs_for_image(
            cell_masks=cell_masks,
            cell_geom_df=cell_geom_df,
            lys_mask=lys_mask,
            prab_mask=prab_mask,
            out_base=png_out_dir / filename_stem,
            filename_stem=filename_stem,
            pad=cfg.cell_png_pad,
            min_cell_area=cfg.cell_png_min_area,
            restrict_to_cell=cfg.restrict_mask_to_cell,
        )

    # Per-cell areas
    per_cell_area_df = compute_per_cell_area_metrics(
        cell_masks=cell_masks,
        nuc_masks=nuc_masks,
        lys_mask=lys_mask,
        prab_mask=prab_mask,
    )

    # Label structures within each cell (global labels + parent map)
    lys_labels_global, lys_parent_cell = label_objects_within_cells(cell_masks, lys_mask)
    prab_labels_global, prab_parent_cell = label_objects_within_cells(cell_masks, prab_mask)

    # Object tables
    lys_props = regionprops_table(
        lys_labels_global,
        intensity_image=lys,
        properties=("label", "area", "min_intensity", "mean_intensity", "max_intensity"),
    )
    lys_df = pd.DataFrame(lys_props)
    if not lys_df.empty:
        lys_df["cell_id"] = lys_df["label"].map(lys_parent_cell)
        lys_df.insert(0, "filename", img_path.name)

    prab_props = regionprops_table(
        prab_labels_global,
        intensity_image=prab,
        properties=("label", "area", "min_intensity", "mean_intensity", "max_intensity"),
    )
    prab_df = pd.DataFrame(prab_props)
    if not prab_df.empty:
        prab_df["cell_id"] = prab_df["label"].map(prab_parent_cell)
        prab_df.insert(0, "filename", img_path.name)

    # Coloc per cell
    cell_coloc_df = compute_cell_coloc(cell_masks, lys, prab, min_pixels=cfg.coloc_min_pixels)
    cell_coloc_df.insert(0, "filename", img_path.name)

    # Merge geometry (centroid + bbox) and areas
    if not cell_coloc_df.empty and not cell_geom_df.empty:
        cell_coloc_df = cell_coloc_df.merge(cell_geom_df, on=["filename", "cell_id"], how="left")
    if not per_cell_area_df.empty and not cell_coloc_df.empty:
        cell_coloc_df = cell_coloc_df.merge(per_cell_area_df, on="cell_id", how="left")

    # Spatial metrics
    lys_spatial_df = compute_per_cell_signal_spatial_metrics(
        cell_masks=cell_masks,
        nuc_masks=nuc_masks,
        signal_mask=lys_mask,
        signal_int=lys,
        prefix="lys",
        perinuclear_radius_px=10.0,
    )
    prab_spatial_df = compute_per_cell_signal_spatial_metrics(
        cell_masks=cell_masks,
        nuc_masks=nuc_masks,
        signal_mask=prab_mask,
        signal_int=prab,
        prefix="prab",
        perinuclear_radius_px=10.0,
    )

    if not cell_coloc_df.empty:
        if not lys_spatial_df.empty:
            cell_coloc_df = cell_coloc_df.merge(lys_spatial_df, on="cell_id", how="left")
        if not prab_spatial_df.empty:
            prab_spatial_df = prab_spatial_df.drop(columns=["nuc_centroid_y", "nuc_centroid_x"], errors="ignore")
            cell_coloc_df = cell_coloc_df.merge(prab_spatial_df, on="cell_id", how="left")

    return {
        "cell_coloc_df": cell_coloc_df,
        "cell_geom_df": cell_geom_df,
        "lys_df": lys_df,
        "prab_df": prab_df,
    }


# ----------------------------
# Batch runner
# ----------------------------
def run_on_folder(
    samplesheet_csv: Path,
    img_dir: Path,
    out_dir: Path,
    pattern: str = "*.tif",
    use_gpu: bool = True,
    cfg: Optional[PipelineConfig] = None,
) -> Dict[str, Path]:
    cfg = cfg or PipelineConfig()
    out_dir.mkdir(parents=True, exist_ok=True)

    samplesheet = pd.read_csv(samplesheet_csv)

    img_lookup = build_img_lookup(img_dir, pattern=pattern)
    samplesheet = add_image_paths(samplesheet, img_lookup)

    present = samplesheet["image_path"].notna()
    missing = samplesheet.loc[~present, "filename"].tolist()
    if missing:
        print(f"[WARN] {len(missing)} filenames not found in folder; skipping first few: {missing[:5]}")
    samplesheet = samplesheet.loc[present].reset_index(drop=True)

    # init model once
    cellpose_model = models.CellposeModel(gpu=use_gpu)

    coloc_all = []
    geom_all = []
    lys_all = []
    prab_all = []
    failures = []

    png_out_dir = out_dir / "cell_pngs"
    png_out_dir.mkdir(parents=True, exist_ok=True)

    for i, row in samplesheet.iterrows():
        try:
            res = process_one_image(row, cfg, cellpose_model, png_out_dir=png_out_dir)

            if not res["cell_coloc_df"].empty:
                coloc_all.append(res["cell_coloc_df"])
            if not res["cell_geom_df"].empty:
                geom_all.append(res["cell_geom_df"])
            if not res["lys_df"].empty:
                lys_all.append(res["lys_df"])
            if not res["prab_df"].empty:
                prab_all.append(res["prab_df"])

        except Exception as e:
            fname = str(row.get("filename", "UNKNOWN"))
            tb = traceback.format_exc()
            failures.append((fname, type(e).__name__, str(e), tb))
            print(f"[FAIL] {fname}: {type(e).__name__}: {e}")
            continue

        if (i + 1) % 10 == 0:
            print(f"Processed {i+1}/{len(samplesheet)} images so far")

    out_paths: Dict[str, Path] = {}

    coloc_all_df = pd.concat(coloc_all, ignore_index=True) if coloc_all else pd.DataFrame()
    geom_all_df  = pd.concat(geom_all,  ignore_index=True) if geom_all  else pd.DataFrame()
    lys_all_df   = pd.concat(lys_all,   ignore_index=True) if lys_all   else pd.DataFrame()
    prab_all_df  = pd.concat(prab_all,  ignore_index=True) if prab_all  else pd.DataFrame()

    out_paths["cells_coloc_all"] = out_dir / "cells_coloc_all.csv"
    out_paths["cells_geometry_all"] = out_dir / "cells_geometry_all.csv"
    out_paths["lys_objects_all"] = out_dir / "lys_objects_all.csv"
    out_paths["prab_objects_all"] = out_dir / "prab_objects_all.csv"

    coloc_all_df.to_csv(out_paths["cells_coloc_all"], index=False)
    geom_all_df.to_csv(out_paths["cells_geometry_all"], index=False)
    lys_all_df.to_csv(out_paths["lys_objects_all"], index=False)
    prab_all_df.to_csv(out_paths["prab_objects_all"], index=False)

    if failures:
        fail_path = out_dir / "failures.csv"
        pd.DataFrame(failures, columns=["filename", "error_type", "error_message", "traceback"]).to_csv(
            fail_path, index=False
        )
        out_paths["failures"] = fail_path
        print(f"[WARN] {len(failures)} failures written to {fail_path}")

    print("Done.")
    return out_paths



Processed 10/30 images so far
Processed 20/30 images so far
Processed 30/30 images so far
Done.
cells_coloc_all -> /Users/kelpschdj/Documents/DataTecnica/Hannah_Bailey/Hannah_RepresentativeImages/pipeline_outputs/cells_coloc_all.csv
cells_geometry_all -> /Users/kelpschdj/Documents/DataTecnica/Hannah_Bailey/Hannah_RepresentativeImages/pipeline_outputs/cells_geometry_all.csv
lys_objects_all -> /Users/kelpschdj/Documents/DataTecnica/Hannah_Bailey/Hannah_RepresentativeImages/pipeline_outputs/lys_objects_all.csv
prab_objects_all -> /Users/kelpschdj/Documents/DataTecnica/Hannah_Bailey/Hannah_RepresentativeImages/pipeline_outputs/prab_objects_all.csv


In [2]:
if __name__ == "__main__":
    samplesheet_csv = Path("/Users/kelpschdj/Documents/DataTecnica/Hannah_Bailey/Hannah_RepresentativeImages/hb_samplesheet.csv")
    img_dir = Path("/Users/kelpschdj/Documents/DataTecnica/Hannah_Bailey/Hannah_RepresentativeImages")
    out_dir = img_dir / "pipeline_outputs"

    cfg = PipelineConfig(
        write_cell_pngs=True,
        write_outlines=True,
        outline_thickness=2,
        write_mask_pngs=True,
        restrict_mask_to_cell=True,
        cell_png_pad=8,
        cell_png_min_area=0,
        bbox_on_crop_edge=True,
    )

    paths = run_on_folder(samplesheet_csv, img_dir, out_dir, pattern="*.tif", use_gpu=True, cfg=cfg)
    for k, v in paths.items():
        print(k, "->", v)


Processed 10/30 images so far
Processed 20/30 images so far
Processed 30/30 images so far
Done.
cells_coloc_all -> /Users/kelpschdj/Documents/DataTecnica/Hannah_Bailey/Hannah_RepresentativeImages/pipeline_outputs/cells_coloc_all.csv
cells_geometry_all -> /Users/kelpschdj/Documents/DataTecnica/Hannah_Bailey/Hannah_RepresentativeImages/pipeline_outputs/cells_geometry_all.csv
lys_objects_all -> /Users/kelpschdj/Documents/DataTecnica/Hannah_Bailey/Hannah_RepresentativeImages/pipeline_outputs/lys_objects_all.csv
prab_objects_all -> /Users/kelpschdj/Documents/DataTecnica/Hannah_Bailey/Hannah_RepresentativeImages/pipeline_outputs/prab_objects_all.csv
